In [ ]:
%pip install numpy pandas matplotlib scikit-learn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/refs/heads/master/data/Telco-Customer-Churn.csv"
df=pd.read_csv(url)
pd.set_option("display.max_columns",None)


In [ ]:
df.head(2)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include='O')

In [ ]:
df['Churn']=df['Churn'].map({'Yes':1,'No':0})

In [ ]:
df['gender']=df['gender'].map({'Female':1,'Male':0})

In [ ]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [ ]:
df['StreamingMovies'].value_counts()

In [ ]:
df.drop(columns='customerID',inplace=True)

In [ ]:
df['PaperlessBilling']=df['PaperlessBilling'].map({ ' Yes':1, "No":0})


In [ ]:
df['StreamingMovies']=df['StreamingMovies'].map({'Yes':1,'No':0,'No internet service':-1})

In [ ]:
df.columns


In [ ]:
numerical_cols=['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges']
for i in numerical_cols:
         if df[i].dtype=="string":
          df[i]=pd.factorize(df[i])[0]

In [ ]:
numerical_cols=[f for f in numerical_cols if f in df.columns]

In [ ]:
df.dropna(inplace=True)
df

In [ ]:
from sklearn.model_selection import train_test_split
X = df[numerical_cols]
y = df['Churn']

#stratify=y
X_train_1, X_test, y_train_1, y_test = train_test_split(X,y,
                                                    test_size=0.20,
                                                    stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train_1)
X_test=scaler.transform(X_test)
print("Max value in X_train:", np.max(X_train))

In [ ]:
x_train,x_val,y_train,y_val=train_test_split(X_train_1,y_train_1,test_size=0.2,random_state=42)

In [ ]:
get_ipython().system('pip install Optuna')

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import optuna

In [ ]:
from sklearn.feature_selection import RFE

In [ ]:

def obj(k):
  n=k.suggest_int('n_estimators',100,500)
  max_depth=k.suggest_int("max_depth",5,50)
  model=RandomForestClassifier(n_estimators=n, max_depth=max_depth)
  rfe=RFE(estimator=model,n_features_to_select=10)
  rfe.fit(X_train_1,y_train_1)
  return rfe.score(x_val,y_val)
study=optuna.create_study(direction="maximize")
study.optimize(obj,n_trials=10)


In [ ]:
best_n_estimators = study.best_params['n_estimators']
best_max_depth = study.best_params['max_depth']

best_model = RandomForestClassifier(n_estimators=best_n_estimators, max_depth=best_max_depth)
rfe = RFE(estimator=best_model, n_features_to_select=10)
rfe.fit(X_train_1, y_train_1)

rfe.support_

In [ ]:
list(zip(numerical_cols,rfe.support_))
selected_features=[feature for feature, selected in zip(numerical_cols,rfe.support_) if selected]
print(selected_features)

In [ ]:
import pickle
with open('rfe.pickle',"wb") as f:
  pickle.dump(rfe,f)

In [ ]:
%pip install tensorflow 
import tensorflow as tf
from tensorflow.keras import layers,models


In [ ]:
def create_ann_model():
  model=models.Sequential([
      layers.Dense(64,activation='relu',input_shape=(10,)),
      layers.Dense(32,activation='relu'),
      layers.Dense(16,activation='relu'),
      layers.Dense(8,activation='relu'),
      layers.Dense(1,activation='sigmoid')
  ])
  return model

In [ ]:
ann_model=create_ann_model()
ann_model.summary()

In [ ]:
ann_model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
X_train_rfe = rfe.transform(X_train_1)
ann_model.fit(X_train_rfe,y_train_1,epochs=32,batch_size=32,validation_split=0.25)

In [ ]:
X_test_rfe=rfe.transform(X_test)
predictions=ann_model.predict(X_test_rfe)
print(predictions.min())

In [ ]:
y_true = y_test
y_pred = (predictions > 0.45).astype(int)
cm = tf.math.confusion_matrix(labels=y_true, predictions=y_pred)
print(cm)

In [ ]:
ann_model.evaluate(X_test_rfe,y_test)

In [ ]:

with open('rfe.pickle', 'rb') as file:
    data = pickle.load(file)
print(data)